# Part 1. Collecting data from the World Bank API
Author: **Dauletkhan Qian**

In this notebook we download two indicators from the World Bank API for 2015-2025:
- `PA.NUS.FCRF` - official exchange rate (local currency per 1 USD, average for the year)
- `FP.CPI.TOTL.ZG` - inflation (%)

We also download the list of countries, because it has region and income group.

Result: `data/wb_countries_raw.csv` and `data/wb_indicators_raw.csv`

## About the data

**Source:** World Bank, *World Development Indicators* (WDI). This is the main World Bank database with statistics
for about 217 countries. The API is free and does not need a key. Documentation:
https://datahelpdesk.worldbank.org/knowledgebase/articles/889392

**Indicators we use**

| Code | Name | Unit | How often | Original source |
|---|---|---|---|---|
| `PA.NUS.FCRF` | Official exchange rate | local currency units per 1 USD | yearly (average of the year) | IMF International Financial Statistics |
| `FP.CPI.TOTL.ZG` | Inflation, consumer prices | % change of prices compared to the previous year | yearly | IMF, national statistics |

**How to read the exchange rate:** the number shows how many units of the local currency you pay for 1 US dollar.
For example, Kazakhstan 2024 ≈ 470 means 1 USD ≈ 470 tenge on average in 2024.
If the number becomes bigger, the currency became **weaker**. Numbers of different countries cannot be compared directly
(1 USD ≈ 0.9 euro but ≈ 15,000 Indonesian rupiah), only the **change** inside one country makes sense.

**How to read inflation:** 5 means that prices in the country grew by 5% during the year.

**Country list** (`/country`): for every country the API gives ISO3 code, name, region
(e.g. "Europe & Central Asia") and income group ("High income", "Upper middle income", "Lower middle income", "Low income").
The income group is decided by the World Bank from GNI per person.

**Period:** 2015-2025 (11 years). We chose 2015 because it is ten full years before the last available year,
and it includes several big events (2020 COVID, 2022 strong dollar).

**What one API record looks like:** one record = one country + one year + one indicator.

In [1]:
import os
import time
import requests
import pandas as pd

os.makedirs("data", exist_ok=True)

BASE_URL = "https://api.worldbank.org/v2"
HEADERS = {"User-Agent": "SIS1 student project"}

First we test one request to see what the API returns.

In [2]:
r = requests.get(BASE_URL + "/country/KAZ/indicator/PA.NUS.FCRF",
                 params={"format": "json", "date": "2024"}, headers=HEADERS)
print(r.status_code)
r.json()

200


[{'page': 1,
  'pages': 1,
  'per_page': 50,
  'total': 1,
  'sourceid': '2',
  'lastupdated': '2026-07-13'},
 [{'indicator': {'id': 'PA.NUS.FCRF',
    'value': 'Official exchange rate (LCU per US$, period average)'},
   'country': {'id': 'KZ', 'value': 'Kazakhstan'},
   'countryiso3code': 'KAZ',
   'date': '2024',
   'value': 468.9625,
   'unit': '',
   'obs_status': '',
   'decimal': 2}]]

The answer is a list with 2 elements: `[info about pages, data]`.
If there is more than one page we need to request them one by one, so we wrote a function for this.

In [3]:
def get_all_pages(url, params):
    all_data = []
    page = 1
    while True:
        params["page"] = page
        r = requests.get(url, params=params, headers=HEADERS, timeout=60)
        r.raise_for_status()
        info, data = r.json()
        if data:
            all_data += data
        print("page", page, "of", info["pages"])
        if page >= info["pages"]:
            break
        page += 1
        time.sleep(0.5)
    return all_data

### Countries

In [4]:
countries_json = get_all_pages(BASE_URL + "/country", {"format": "json", "per_page": 500})

rows = []
for c in countries_json:
    rows.append({
        "iso3": c["id"],
        "name": c["name"],
        "region": c["region"]["value"],
        "income_level": c["incomeLevel"]["value"],
    })

countries = pd.DataFrame(rows)
print(countries.shape)
countries.head()

page 1 of 1
(296, 4)


,iso3,name,region,income_level
0,ABW,Aruba,Latin America & Caribbean,High income
1,AFE,Africa Eastern and Southern,Aggregates,Aggregates
2,AFG,Afghanistan,"Middle East, North Africa, Afghanistan & Pakistan",Low income
3,AFR,Africa,Aggregates,Aggregates
4,AFW,Africa Western and Central,Aggregates,Aggregates


### Exchange rate and inflation

In [5]:
indicators = {"PA.NUS.FCRF": "exchange_rate", "FP.CPI.TOTL.ZG": "inflation"}

rows = []
for code, name in indicators.items():
    url = BASE_URL + "/country/all/indicator/" + code
    data = get_all_pages(url, {"format": "json", "date": "2015:2025", "per_page": 1000})
    for item in data:
        rows.append({
            "iso3": item["countryiso3code"],
            "name": item["country"]["value"],
            "year": item["date"],
            "indicator": name,
            "value": item["value"],
        })

wb = pd.DataFrame(rows)
print(wb.shape)
wb.head()

page 1 of 3


page 2 of 3


page 3 of 3
page 1 of 3


page 2 of 3


page 3 of 3
(5830, 5)


,iso3,name,year,indicator,value
0,AFE,Africa Eastern and Southern,2025,exchange_rate,NaN
1,AFE,Africa Eastern and Southern,2024,exchange_rate,NaN
2,AFE,Africa Eastern and Southern,2023,exchange_rate,NaN
3,AFE,Africa Eastern and Southern,2022,exchange_rate,NaN
4,AFE,Africa Eastern and Southern,2021,exchange_rate,NaN


### Quick look at the data

In [6]:
wb.info()

<class 'pandas.DataFrame'>
RangeIndex: 5830 entries, 0 to 5829
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   iso3       5830 non-null   str    
 1   name       5830 non-null   str    
 2   year       5830 non-null   str    
 3   indicator  5830 non-null   str    
 4   value      4761 non-null   float64
dtypes: float64(1), str(4)
memory usage: 227.9 KB


In [7]:
print("missing values:")
print(wb.groupby("indicator")["value"].apply(lambda x: x.isna().sum()))
print()
print(countries["region"].value_counts())

missing values:
indicator
exchange_rate    647
inflation        422
Name: value, dtype: int64

region
Aggregates                                           79
Europe & Central Asia                                58
Sub-Saharan Africa                                   48
Latin America & Caribbean                            42
East Asia & Pacific                                  37
Middle East, North Africa, Afghanistan & Pakistan    23
South Asia                                            6
North America                                         3
Name: count, dtype: int64


## Understanding the data
Now we look at the data more carefully to understand what we have.

In [8]:
aggregates = countries[countries["region"] == "Aggregates"]
real = countries[countries["region"] != "Aggregates"]
print("entries in the country list:", len(countries))
print("  real countries:", len(real))
print("  groups (aggregates):", len(aggregates), "for example", list(aggregates["name"].head(5)))
print()
print("years:", sorted(wb["year"].unique()))
print("rows per indicator:")
print(wb["indicator"].value_counts())

entries in the country list: 296
  real countries: 217
  groups (aggregates): 79 for example ['Africa Eastern and Southern', 'Africa', 'Africa Western and Central', 'Arab World', 'East Asia & Pacific (IBRD-only countries)']

years: ['2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024', '2025']
rows per indicator:
indicator
exchange_rate    2915
inflation        2915
Name: count, dtype: int64


How many countries have a value in each year (only real countries, without groups):

In [9]:
wb_real = wb[wb["iso3"].isin(real["iso3"])]
per_year = wb_real.dropna(subset=["value"]).pivot_table(index="year", columns="indicator",
                                                        values="value", aggfunc="count")
per_year

indicator,exchange_rate,inflation
year,,
2015,210,189
2016,210,189
2017,210,184
2018,208,183
2019,208,183
2020,209,178
2021,207,178
2022,207,178
2023,205,176


Data for 2025 is not complete yet - the World Bank adds new values during the year.

In [10]:
print("Exchange rate (local currency per 1 USD):")
print(wb_real[wb_real["indicator"] == "exchange_rate"]["value"].describe().round(2))
print()
print("Inflation, %:")
print(wb_real[wb_real["indicator"] == "inflation"]["value"].describe().round(2))

Exchange rate (local currency per 1 USD):
count     2257.00
mean       822.00
std       4297.08
min          0.30
25%          1.00
50%          7.99
75%        128.87
max      89500.00
Name: value, dtype: float64

Inflation, %:
count    1977.00
mean        7.07
std        23.62
min        -6.69
25%         1.24
50%         3.03
75%         6.15
max       557.20
Name: value, dtype: float64


The exchange rate goes from less than 1 (for example euro, British pound, Kuwaiti dinar) to tens of thousands
(Lebanon, Iran, Indonesia), because every country has its own currency. Inflation is mostly a few percent,
but some countries have huge values (hyperinflation).

Example for some countries:

In [11]:
examples = ["KAZ", "USA", "DEU", "JPN", "TUR", "SAU"]
ex = wb_real[(wb_real["indicator"] == "exchange_rate") & (wb_real["iso3"].isin(examples))]
ex.pivot(index="name", columns="year", values="value").round(2)

year,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
name,,,,,,,,,,,
Germany,0.90,0.90,0.89,0.85,0.89,0.88,0.85,0.95,0.92,0.92,0.88
Japan,121.04,108.79,112.17,110.42,109.01,106.77,109.75,131.50,140.49,151.37,149.66
Kazakhstan,221.73,342.16,326.00,344.71,382.75,412.95,425.91,460.16,456.16,468.96,521.04
Saudi Arabia,3.75,3.75,3.75,3.75,3.75,3.75,3.75,3.75,3.75,3.75,3.75
Turkiye,2.72,3.02,3.65,4.83,5.67,7.01,8.85,16.55,23.74,32.81,39.45
United States,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00


- **United States** = 1 every year (dollar to dollar), we will remove it later.
- **Germany** - euro, around 0.85-0.95 euro per dollar.
- **Saudi Arabia** - 3.75 every year, the riyal is fixed to the dollar.
- **Turkey** - from 2.7 to almost 40, the lira lost most of its value.
- **Kazakhstan** - tenge became weaker after 2015.

In [12]:
ex_inf = wb_real[(wb_real["indicator"] == "inflation") & (wb_real["iso3"].isin(examples))]
ex_inf.pivot(index="name", columns="year", values="value").round(1)

year,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
name,,,,,,,,,,,
Germany,0.5,0.5,1.5,1.7,1.4,0.1,3.1,6.9,5.9,2.3,2.2
Japan,0.8,-0.1,0.5,1.0,0.5,-0.0,-0.2,2.5,3.3,2.7,3.2
Kazakhstan,6.7,14.4,7.4,6.2,5.3,6.7,8.0,15.0,14.5,8.7,11.4
Saudi Arabia,1.2,2.1,-0.8,2.5,-1.2,3.4,3.1,2.5,2.3,1.7,2.1
Turkiye,7.7,7.8,11.1,16.3,15.2,12.3,19.6,72.3,53.9,58.5,34.9
United States,0.1,1.3,2.1,2.4,1.8,1.2,4.7,8.0,4.1,2.9,NaN


In [13]:
print("countries by income group:")
print(real["income_level"].value_counts())
print()
print("countries by region:")
print(real["region"].value_counts())

countries by income group:
income_level
High income            86
Upper middle income    59
Lower middle income    47
Low income             25
Name: count, dtype: int64

countries by region:
region
Europe & Central Asia                                58
Sub-Saharan Africa                                   48
Latin America & Caribbean                            42
East Asia & Pacific                                  37
Middle East, North Africa, Afghanistan & Pakistan    23
South Asia                                            6
North America                                         3
Name: count, dtype: int64


What we noticed:
- there are not only countries but also groups like "World", "Euro area", "Low income" (region = "Aggregates")
- a lot of missing values, especially for 2025
- `year` is a string

We leave this for Part 3 (cleaning) and just save the raw data.

### Files we save

`data/wb_countries_raw.csv` - one row per country or group

| Column | Meaning |
|---|---|
| iso3 | ISO code of the country (KAZ, DEU, ...) |
| name | name used by the World Bank |
| region | region, or "Aggregates" for groups |
| income_level | World Bank income group |

`data/wb_indicators_raw.csv` - one row = country + year + indicator

| Column | Meaning |
|---|---|
| iso3 | ISO code |
| name | country name |
| year | year (2015-2025) |
| indicator | `exchange_rate` or `inflation` |
| value | the number (empty if no data) |

In [14]:
countries.to_csv("data/wb_countries_raw.csv", index=False)
wb.to_csv("data/wb_indicators_raw.csv", index=False)
print("saved")

saved
